# PRUDENCIA — Deep Learning niveau 1

Ce notebook présente le problème de classification du risque AI Act, explore le dataset, construit une **baseline explicable**, puis prépare une comparaison équitable avec **JuriBERT** et **CamemBERT**. Les modèles utilisent la même séparation afin que leurs scores soient comparables.

> Le script `.py` reste volontairement simple. Les analyses avancées destinées à la soutenance sont regroupées ici.

## 1. Imports et configuration

In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             classification_report, f1_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
TEXT_COLUMN = "Q3"
LABEL_COLUMN = "risk_level_aiact"
DATASET_PATH = Path("../datasets/dl/dl_juribert_training_cases.csv")
OUTPUT_DIR = Path("../outputs/dl_niveau_1_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

## 2. Chargement et contrôle du dataset

Le séparateur est `;` et le fichier est désormais en UTF-8. Le texte descriptif `Q3` est l'entrée du modèle et `risk_level_aiact` est la classe à prédire.

In [ ]:
df_brut = pd.read_csv(DATASET_PATH, sep=";", encoding="utf-8")
print(f"Dimensions : {df_brut.shape[0]} lignes × {df_brut.shape[1]} colonnes")
display(df_brut.head(3))
display(pd.DataFrame({"type": df_brut.dtypes, "manquants": df_brut.isna().sum(),
                      "valeurs_uniques": df_brut.nunique()}))

## 3. Nettoyage et qualité des données

In [ ]:
df = df_brut[["id", "titre_cas", TEXT_COLUMN, LABEL_COLUMN]].copy()
avant = len(df)
df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(str).str.strip()
df = df[(df[TEXT_COLUMN] != "") & (df[LABEL_COLUMN] != "")]
doublons = df.duplicated(subset=[TEXT_COLUMN, LABEL_COLUMN]).sum()
df = df.drop_duplicates(subset=[TEXT_COLUMN, LABEL_COLUMN]).reset_index(drop=True)
print(f"Lignes initiales : {avant}")
print(f"Doublons retirés : {doublons}")
print(f"Lignes utilisables : {len(df)}")

## 4. EDA — répartition de la cible

Cette analyse vérifie si les classes sont équilibrées. Une accuracy élevée peut être trompeuse lorsque certaines classes sont rares ; la **macro-F1** donnera donc le même poids à chaque niveau de risque.

In [ ]:
repartition = df[LABEL_COLUMN].value_counts().rename_axis("classe").reset_index(name="nombre")
repartition["pourcentage"] = 100 * repartition["nombre"] / len(df)
display(repartition)

ax = sns.barplot(data=repartition, x="classe", y="nombre", hue="classe", legend=False)
ax.set(title="Répartition des niveaux de risque", xlabel="Niveau de risque", ylabel="Nombre de cas")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## 5. EDA — longueur et vocabulaire des textes

In [ ]:
df["nombre_caracteres"] = df[TEXT_COLUMN].str.len()
df["nombre_mots"] = df[TEXT_COLUMN].str.split().str.len()
display(df[["nombre_caracteres", "nombre_mots"]].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["nombre_mots"], bins=20, ax=axes[0])
axes[0].set_title("Distribution de la longueur des textes")
sns.boxplot(data=df, x=LABEL_COLUMN, y="nombre_mots", hue=LABEL_COLUMN,
            legend=False, ax=axes[1])
axes[1].set_title("Longueur selon le niveau de risque")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
mots = (df[TEXT_COLUMN].str.lower().str.findall(r"[a-zà-ÿ]{4,}").explode()
        .value_counts().head(20).rename_axis("mot").reset_index(name="frequence"))
display(mots)
sns.barplot(data=mots, y="mot", x="frequence", hue="mot", legend=False)
plt.title("Mots fréquents du corpus (analyse descriptive)")
plt.tight_layout()
plt.show()

## 6. Séparation commune des données

La stratification conserve la proportion des classes. `random_state=42` fixe la séparation. Tous les modèles ci-dessous utilisent exactement les mêmes observations d'entraînement et de validation.

In [ ]:
X_train, X_validation, y_train, y_validation = train_test_split(
    df[TEXT_COLUMN], df[LABEL_COLUMN], test_size=0.20,
    random_state=RANDOM_STATE, stratify=df[LABEL_COLUMN])
print("Entraînement :", len(X_train), "— Validation :", len(X_validation))
display(pd.crosstab(index=y_train, columns="train"))
display(pd.crosstab(index=y_validation, columns="validation"))

## 7. Modèle naïf de référence

Le `DummyClassifier` prédit uniquement selon la stratégie choisie. Il vérifie qu'un vrai modèle apprend davantage que la simple classe majoritaire.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(np.zeros((len(X_train), 1)), y_train)
pred_dummy = dummy.predict(np.zeros((len(X_validation), 1)))
resultats = [{"modele": "Classe majoritaire",
              "exactitude": accuracy_score(y_validation, pred_dummy),
              "macro_f1": f1_score(y_validation, pred_dummy, average="macro", zero_division=0)}]
pd.DataFrame(resultats)

## 8. Baseline NLP — TF-IDF + régression logistique

TF-IDF transforme les mots et groupes de deux mots en variables numériques. La régression logistique apprend ensuite à associer ces indices lexicaux aux classes. Cette baseline est rapide, compréhensible et indispensable pour vérifier si le coût d'un Transformer apporte un gain réel.

In [ ]:
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2,
                              max_features=5000, sublinear_tf=True)),
    ("classification", LogisticRegression(max_iter=2000, class_weight="balanced",
                                           random_state=RANDOM_STATE)),
])
baseline.fit(X_train, y_train)
pred_baseline = baseline.predict(X_validation)
resultats.append({"modele": "TF-IDF + régression logistique",
                  "exactitude": accuracy_score(y_validation, pred_baseline),
                  "macro_f1": f1_score(y_validation, pred_baseline, average="macro", zero_division=0)})
display(pd.DataFrame(resultats).round(3))
print(classification_report(y_validation, pred_baseline, zero_division=0))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_validation, pred_baseline, xticks_rotation=25, cmap="Blues")
plt.title("Matrice de confusion — baseline TF-IDF")
plt.tight_layout()
plt.show()

## 9. Interprétation de la baseline

Les coefficients de la régression logistique indiquent les termes les plus associés à chaque classe. Il s'agit d'une **association statistique**, et non d'une règle juridique absolue.

In [ ]:
termes = baseline.named_steps["tfidf"].get_feature_names_out()
classifieur = baseline.named_steps["classification"]
for indice, classe in enumerate(classifieur.classes_):
    meilleurs = np.argsort(classifieur.coef_[indice])[-8:][::-1]
    print(f"{classe:12} :", ", ".join(termes[meilleurs]))

## 10. Fine-tuning comparatif des Transformers (cellules optionnelles)

Les modèles comparés sont :

- `dascim/juribert-base` : modèle juridique français principal ;
- `camembert-base` : modèle généraliste français servant de comparaison.

Leur fine-tuning est désactivé par défaut car il télécharge des poids volumineux et peut prendre du temps sur Mac. Mettre `ENTRAINER_TRANSFORMERS = True` pour lancer l'expérience. La même séparation et les mêmes métriques sont utilisées.

In [ ]:
ENTRAINER_TRANSFORMERS = False
MODELES_TRANSFORMERS = {
    "JuriBERT": "dascim/juribert-base",
    "CamemBERT": "camembert-base",
}
MAX_LENGTH = 256
EPOCHS = 3
BATCH_SIZE = 4

In [ ]:
if ENTRAINER_TRANSFORMERS:
    from datasets import Dataset
    from sklearn.preprocessing import LabelEncoder
    from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                              DataCollatorWithPadding, Trainer, TrainingArguments)

    encodeur = LabelEncoder().fit(df[LABEL_COLUMN])
    id2label = {i: str(c) for i, c in enumerate(encodeur.classes_)}
    label2id = {c: i for i, c in id2label.items()}

    def entrainer_transformer(nom_affiche, identifiant):
        tokenizer = AutoTokenizer.from_pretrained(identifiant)
        train_table = pd.DataFrame({TEXT_COLUMN: X_train,
                                    "labels": encodeur.transform(y_train)})
        valid_table = pd.DataFrame({TEXT_COLUMN: X_validation,
                                    "labels": encodeur.transform(y_validation)})
        train_ds = Dataset.from_pandas(train_table, preserve_index=False)
        valid_ds = Dataset.from_pandas(valid_table, preserve_index=False)
        def tokeniser(lot):
            return tokenizer(lot[TEXT_COLUMN], truncation=True, max_length=MAX_LENGTH)
        train_ds = train_ds.map(tokeniser, batched=True, remove_columns=[TEXT_COLUMN])
        valid_ds = valid_ds.map(tokeniser, batched=True, remove_columns=[TEXT_COLUMN])
        modele = AutoModelForSequenceClassification.from_pretrained(
            identifiant, num_labels=len(id2label), id2label=id2label,
            label2id=label2id, ignore_mismatched_sizes=True)
        args = TrainingArguments(
            output_dir=str(OUTPUT_DIR / nom_affiche.lower()), eval_strategy="epoch",
            save_strategy="epoch", learning_rate=2e-5, num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE, load_best_model_at_end=True,
            metric_for_best_model="eval_loss", save_total_limit=1,
            report_to="none", seed=RANDOM_STATE)
        def metriques(eval_pred):
            logits, labels = eval_pred
            predictions = np.argmax(logits, axis=-1)
            return {"exactitude": accuracy_score(labels, predictions),
                    "macro_f1": f1_score(labels, predictions, average="macro",
                                         zero_division=0)}
        trainer = Trainer(model=modele, args=args, train_dataset=train_ds,
                          eval_dataset=valid_ds, processing_class=tokenizer,
                          data_collator=DataCollatorWithPadding(tokenizer),
                          compute_metrics=metriques)
        trainer.train()
        sortie = trainer.predict(valid_ds)
        predictions = np.argmax(sortie.predictions, axis=-1)
        return {"modele": nom_affiche,
                "exactitude": accuracy_score(encodeur.transform(y_validation), predictions),
                "macro_f1": f1_score(encodeur.transform(y_validation), predictions,
                                     average="macro", zero_division=0)}

    for nom, identifiant in MODELES_TRANSFORMERS.items():
        resultats.append(entrainer_transformer(nom, identifiant))

## 11. Tableau comparatif des modèles

In [ ]:
comparaison = pd.DataFrame(resultats).sort_values("macro_f1", ascending=False)
display(comparaison.style.format({"exactitude": "{:.3f}", "macro_f1": "{:.3f}"}))

comparaison_longue = comparaison.melt(id_vars="modele", var_name="metrique", value_name="score")
sns.barplot(data=comparaison_longue, x="modele", y="score", hue="metrique")
plt.ylim(0, 1)
plt.title("Comparaison sur la même validation")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()
comparaison.to_csv(OUTPUT_DIR / "comparaison_modeles.csv", index=False)

## 12. Analyse des erreurs de la baseline

In [ ]:
erreurs = pd.DataFrame({"texte": X_validation.values,
                         "classe_reelle": y_validation.values,
                         "classe_predite": pred_baseline})
erreurs = erreurs[erreurs["classe_reelle"] != erreurs["classe_predite"]]
print(f"Nombre d'erreurs : {len(erreurs)}")
display(erreurs.head(10))

## 13. Conclusion pour la soutenance

- L'EDA vérifie la qualité, le déséquilibre des classes et la longueur des textes.
- Le modèle naïf donne un seuil minimal à dépasser.
- TF-IDF + régression logistique fournit une baseline rapide et interprétable.
- JuriBERT exploite un préentraînement spécialisé dans le français juridique.
- CamemBERT permet de vérifier si la spécialisation juridique apporte réellement un gain.
- La comparaison est valide uniquement si les modèles utilisent la même séparation et les mêmes métriques.
- Sur seulement 163 cas, les scores restent sensibles à la séparation : ils doivent être présentés avec prudence.

**Question centrale :** le gain éventuel de JuriBERT justifie-t-il son coût d'entraînement par rapport à la baseline ?